In [1]:
from granuscore.claim_splitter import SpacyNounPhraseSplitter
from granuscore import GranuScore

scorer = GranuScore(claim_splitter=SpacyNounPhraseSplitter(skip_brackets=True,
                                                           skip_figures=True,
                                                           skip_latex=True,
                                                           skip_arxiv_meta=True,
                                                           skip_pdf_noise=True))

[GranuScorer] macOS detected → using FAISS single-thread mode.


### Calc effect size of significance

In [2]:
from granuscore.loader import JSONLineReader
from evaluation.config import PROJECT_DIR

data_path = f"{PROJECT_DIR}/data/s2orc/filtered_papers.jsonl"
papers = JSONLineReader().read(data_path)


intro = scorer.predict(
        [p['introduction_par'] for p in papers],
        show_progress_bar=True,
    )

rel = scorer.predict(
        [p['related_work_par'] for p in papers],
        show_progress_bar=True,
    )

Granuscore: 100%|██████████| 8/8 [00:53<00:00,  6.69s/it]


In [5]:
from scipy.stats import ttest_rel, wilcoxon
import numpy as np

diffs = intro - rel
cohen_dz = diffs.mean() / diffs.std()

{
    "correct_ordering": np.mean(intro > rel),
    "intro_mean": np.mean(intro),
    "intro_std": np.std(intro),
    "rel_mean": np.mean(rel),
    "rel_std": np.std(rel),
    "cohen_dz": cohen_dz,
    "ttest": ttest_rel(intro, rel),
    "wilcoxon": wilcoxon(intro, rel, alternative="greater"),
}

{'correct_ordering': np.float64(0.6871165644171779),
 'intro_mean': np.float32(75.28658),
 'intro_std': np.float32(4.9420567),
 'rel_mean': np.float32(72.42978),
 'rel_std': np.float32(5.5423822),
 'cohen_dz': np.float32(0.4240001),
 'ttest': TtestResult(statistic=np.float64(13.252971205919893), pvalue=np.float64(5.481267604952616e-37), df=np.int64(977)),
 'wilcoxon': WilcoxonResult(statistic=np.float64(353612.0), pvalue=np.float64(5.723240976808826e-39))}